# 🔱 Kronos Fine-tune — Binance Perpétuos

Este notebook treina o Kronos em dados reais de perpétuos da Binance.

**Pré-requisitos:**
1. `Runtime → Change runtime type → T4 GPU` (gratuito) ou A100 (Colab Pro)
2. Upload dos CSVs gerados localmente para o Google Drive (pasta `Kronos/data/`)

**Estimativa de tempo (T4 GPU):**
- Tokenizer: ~15 min
- Predictor: ~25 min
- Total: ~40 min por ativo


## 1. Monta Google Drive (salva checkpoints entre sessões)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Kronos'
os.makedirs(f'{DRIVE_ROOT}/data',      exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/finetuned', exist_ok=True)
print('✅ Drive montado em', DRIVE_ROOT)

## 2. Clona o repositório Kronos

In [ ]:
import os

REPO_DIR = '/content/Kronos'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shiyu-coder/Kronos.git {REPO_DIR}
else:
    print('Repo já existe, atualizando...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('✅ Diretório atual:', os.getcwd())

## 3. Instala dependências

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ccxt

# Verifica GPU
import torch
print(f'GPU disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Dispositivo: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Opção A — Baixa dados direto da Binance (recomendado)
Não precisa de upload manual. Faz o download aqui mesmo.

In [ ]:
# ── Configure aqui ─────────────────────────────────────────────────────
SYMBOL   = 'BTC'    # BTC | ETH | SOL
INTERVAL = '1h'
DAYS     = 1000
# ───────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, '/content/Kronos')
from infra.binance_data_pipeline import build_exchange, fetch_symbol_history, save_csv

exchange = build_exchange()
df = fetch_symbol_history(exchange, SYMBOL, INTERVAL, DAYS)
csv_path = save_csv(df, SYMBOL, INTERVAL)

# Copia para o Drive (persistência entre sessões)
import shutil
drive_csv = f'{DRIVE_ROOT}/data/BN_{SYMBOL}_{INTERVAL}.csv'
shutil.copy(csv_path, drive_csv)
print(f'✅ Backup no Drive: {drive_csv}')

## 4. Opção B — Upload manual do CSV gerado localmente
Use se já rodou o pipeline localmente.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # selecione BN_BTC_1h.csv
#
# import shutil
# for fname in uploaded:
#     shutil.move(fname, f'/content/Kronos/finetune_csv/data/{fname}')
#     print(f'Movido: {fname}')

## 5. Gera config de fine-tune e ajusta paths para o Colab

In [ ]:
import yaml, os

SYMBOL   = 'BTC'   # deve bater com o da célula 4
INTERVAL = '1h'

config = {
    'data': {
        'data_path':       f'/content/Kronos/finetune_csv/data/BN_{SYMBOL}_{INTERVAL}.csv',
        'lookback_window': 512,
        'predict_window':  24,
        'max_context':     512,
        'clip':            5.0,
        'train_ratio':     0.85,
        'val_ratio':       0.10,
        'test_ratio':      0.05,
    },
    'training': {
        'tokenizer_epochs':        30,
        'basemodel_epochs':        20,
        'batch_size':              32,
        'log_interval':            50,
        'num_workers':             2,
        'seed':                    42,
        'tokenizer_learning_rate': 2e-4,
        'predictor_learning_rate': 1e-6,
        'adam_beta1':              0.9,
        'adam_beta2':              0.95,
        'adam_weight_decay':       0.1,
        'accumulation_steps':      1,
    },
    'model_paths': {
        'pretrained_tokenizer': 'NeoQuasar/Kronos-Tokenizer-base',
        'pretrained_predictor': 'NeoQuasar/Kronos-small',
        'exp_name':             f'BN_{SYMBOL}_{INTERVAL}',
        # Salva direto no Drive — sobrevive ao fim da sessão
        'base_path':            f'{DRIVE_ROOT}/finetuned',
        'base_save_path':       '',
        'finetuned_tokenizer':  '',
        'tokenizer_save_name':  'tokenizer',
        'basemodel_save_name':  'basemodel',
    },
    'experiment': {
        'name':                f'kronos_bn_{SYMBOL.lower()}_{INTERVAL}',
        'description':         f'Kronos fine-tuned on Binance {SYMBOL}/USDT perp {INTERVAL} (Colab)',
        'use_comet':           False,
        'train_tokenizer':     True,
        'train_basemodel':     True,
        'skip_existing':       False,
        'pre_trained_tokenizer': True,
        'pre_trained_predictor': True,
    },
    'device': {'use_cuda': True, 'device_id': 0},
}

cfg_path = f'/content/Kronos/finetune_csv/configs/colab_{SYMBOL.lower()}_{INTERVAL}.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✅ Config salvo em: {cfg_path}')
print(f'   Checkpoints vão para: {DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/')

## 6. Treina o Kronos 🚀

In [ ]:
import subprocess, sys

SYMBOL   = 'BTC'
INTERVAL = '1h'
cfg_path = f'/content/Kronos/finetune_csv/configs/colab_{SYMBOL.lower()}_{INTERVAL}.yaml'

cmd = [
    sys.executable,
    '/content/Kronos/finetune_csv/train_sequential.py',
    '--config', cfg_path
]

print('Iniciando treinamento...')
print(f'Comando: {" ".join(cmd)}\n')

result = subprocess.run(cmd, cwd='/content/Kronos')

if result.returncode == 0:
    print('\n✅ Treinamento concluído!')
    print(f'Modelo salvo em: {DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/')
else:
    print('\n❌ Erro no treinamento — veja o output acima')

## 7. Valida o modelo treinado

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '/content/Kronos')

from model import Kronos, KronosTokenizer, KronosPredictor

SYMBOL   = 'BTC'
INTERVAL = '1h'
tok_path  = f'{DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/tokenizer/best_model'
pred_path = f'{DRIVE_ROOT}/finetuned/BN_{SYMBOL}_{INTERVAL}/basemodel/best_model'

tokenizer = KronosTokenizer.from_pretrained(tok_path)
model     = Kronos.from_pretrained(pred_path)
predictor = KronosPredictor(model, tokenizer, device='cuda', max_context=512)

# Testa com os últimos 200 candles do CSV
df  = pd.read_csv(f'/content/Kronos/finetune_csv/data/BN_{SYMBOL}_{INTERVAL}.csv')
df['timestamps'] = pd.to_datetime(df['timestamps'])

x_df = df.tail(200)[['open','high','low','close','volume']].reset_index(drop=True)
x_ts = df.tail(200)['timestamps'].reset_index(drop=True)

from datetime import timedelta
y_ts = pd.Series([x_ts.iloc[-1] + timedelta(hours=i+1) for i in range(24)])

forecast = predictor.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
                             pred_len=24, T=1.0, top_p=0.9, sample_count=3)

print(f'Close atual : ${df["close"].iloc[-1]:,.2f}')
print(f'Forecast +24h: ${forecast["close"].iloc[-1]:,.2f}')
print(forecast[['close','high','low']].head(6))